## Stage 1: Load & Clean

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

df = pd.read_csv('/Users/apple/Downloads/Real estate.csv')

df.drop(columns=['No', 'X1 transaction date'], inplace=True)

df.columns = ['house_age', 'distance_to_mrt', 'convenience_stores',
              'latitude', 'longitude', 'price_per_area']

df = df.apply(pd.to_numeric, errors='coerce')
df.dropna(inplace=True)

print(f"Clean dataset: {len(df):,} rows, {df.shape[1]-1} features -> target: price_per_area\n")
print(df.describe().round(2))

## Stage 2: Correlation & Heatmap

In [ ]:
corr = df.corr()
print("Correlation with price_per_area:")
print(corr['price_per_area'].sort_values(ascending=False).to_string())
print()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('/Users/apple/reg-ana/re_correlation_heatmap.png', dpi=150)
plt.close()
print("Saved: re_correlation_heatmap.png")

## Stage 3: Visualisations

In [ ]:
# Histogram — distribution of price per area
plt.figure(figsize=(8, 5))
plt.hist(df['price_per_area'], bins=40, color='steelblue', edgecolor='white')
plt.title('Distribution of House Price per Unit Area')
plt.xlabel('Price per Unit Area')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('/Users/apple/reg-ana/re_histogram.png', dpi=150)
plt.close()
print("Saved: re_histogram.png")

# Bar plot — avg price by number of convenience stores
store_avg = df.groupby('convenience_stores')['price_per_area'].mean()
plt.figure(figsize=(8, 5))
store_avg.plot(kind='bar', color='coral', edgecolor='white')
plt.title('Average Price per Area by Number of Convenience Stores')
plt.xlabel('Number of Convenience Stores')
plt.ylabel('Avg Price per Unit Area')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('/Users/apple/reg-ana/re_barplot.png', dpi=150)
plt.close()
print("Saved: re_barplot.png")

## Stage 4: Train / Test Split & Model Training

In [ ]:
FEATURES = ['house_age', 'distance_to_mrt', 'convenience_stores', 'latitude', 'longitude']
TARGET = 'price_per_area'

X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}\n")

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree':     DecisionTreeRegressor(random_state=42),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Trained: {name}")

## Stage 5: Model Evaluation

In [ ]:
header = f"{'Model':<22} {'R2 Score':>10} {'MSE':>12} {'MAE':>10}"
print(header)
print('-' * len(header))

for name, model in models.items():
    preds = model.predict(X_test)
    r2  = r2_score(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    print(f"{name:<22} {r2:>10.4f} {mse:>12.4f} {mae:>10.4f}")

print("\nDone. All plots saved to /Users/apple/reg-ana/")